In [2]:
import tensorflow
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd

In [7]:
model=load_model('model.h5')
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo=pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)


In [8]:
# Example input
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


In [9]:
df = pd.DataFrame([input_data])
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [10]:
# Encode Gender (Label Encoding)
df['Gender'] = label_encoder_gender.transform(df['Gender'])

In [11]:
# Encode Geography (One-Hot Encoding)
geo_encoded = onehot_encoder_geo.transform(df[['Geography']]).toarray()
geo_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

# Drop original Geography column and concat encoded
df = df.drop('Geography', axis=1)
df = pd.concat([df, geo_df], axis=1)


In [12]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
# Scale input
df_scaled = scaler.transform(df)


In [14]:
##Predict churn
prediction=model.predict(df_scaled)
prediction

1/1 [==============================] - 0s 111ms/step


array([[0.05687192]], dtype=float32)

In [15]:
prediction_proba=prediction[0][0]

In [16]:
prediction_proba

0.056871917

In [17]:
if prediction_proba>0.5:
    print("The person is likely to churn")
else:
    print("The person is unlikely to churn")

The person is unlikely to churn
